In [1]:
#IMPORTING stuff including my online repo with useful tools for processing chess files (PGN)

import sys
import os, sys
!pip install chess mlflow
if os.path.exists("/kaggle/working/ludus_scacchi"):
    print("Repository already exists, skipping clone.")
    sys.path.append("/kaggle/working/ludus_scacchi")
else:
    !git clone https://github.com/paolomostardi/ludus_scacchi.git /kaggle/working/ludus_scacchi
    sys.path.append("/kaggle/working/ludus_scacchi")
from pathlib import Path
from Backend.data_pipeline import from_PGN_generate_bitboards
from Backend.data_pipeline import gm_pipeline as pipe
import pandas as pd
import numpy as np
import keras
import mlflow
import mlflow.keras

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "file:///kaggle/working/mlruns")
if MLFLOW_TRACKING_URI.startswith("file://"):
    Path(MLFLOW_TRACKING_URI.replace("file://", "")).mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("ludus_cnn")
mlflow.keras.autolog(log_models=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 49.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.5 MB/s eta 0:00:00
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=

2026-03-28 11:03:32.700445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774695812.916947      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774695812.980591      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774695813.469288      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774695813.469342      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774695813.469351      24 computation_placer.cc:177] computation placer alr

In [2]:
# using a testing model to train faster and experiment with mlflow

from Backend.model_architecture.implementation.experimental.model_testing_mlflow import model_testing_mlflow
model = model_testing_mlflow(conv_filters=12, upsample_factor=4)

I0000 00:00:1774695839.281789      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [3]:
# deleting the folder to avoid chaos in the output
!rm -r ludus_scacchi

In [4]:
from keras.models import load_model

In [5]:
# how many cycles need to be done
# used to avoid timeout with heavier models 

MAX_N = 5
n = 5
starting_n = 0
conv_size = 32

model_name = 'lenet_baseline'
year_month = '2013-01'

mlflow_run_id = None

with mlflow.start_run(run_name=f"{model_name}_{year_month}") as run:
    mlflow_run_id = run.info.run_id
    mlflow.log_params({
        "model_name": model_name,
        "year_month": year_month,
        "max_n": MAX_N,
        "start_chunk": starting_n,
        "end_chunk": n - 1,
        "conv_size": conv_size,
        "kernel_size": (2, 2),
        "epochs_per_chunk": 3,
        "batch_size": 64
    })

    for i in range(starting_n, n):
        print("=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=")
        print("Starting training with chunk number: ", i)
        
        model.name = model_name + '_chunk_' + str(i) + '_' + year_month
        x = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '.npy')
        y = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '_y.npy')

        # leaving space for evaluation 
        if i == MAX_N - 1:
            x = x[:-10000]
            y = y[:-10000]        
        y2 = []
        for j in y:
            y2.append(j[0].flat)
        y2 = np.array(y2)
        
        history = model.fit(x, y2, batch_size=64, epochs=3)
        mlflow.log_metric("chunk_index", i, step=i)
        model.save(model.name + '.keras')
        print("saved model with name: ", model.name)
    model.save(f'{model_name}.keras')


=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  0


Epoch 1/3


I0000 00:00:1774695855.023847     100 service.cc:152] XLA service 0x782464004a80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774695855.023896     100 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774695855.433355     100 cuda_dnn.cc:529] Loaded cuDNN version 91002


   36/15625 ━━━━━━━━━━━━━━━━━━━━ 1:09 4ms/step - accuracy: 0.0111 - loss: 4.1929

I0000 00:00:1774695858.454699     100 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

2026/03/28 11:18:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_3_2013-01
=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  4


Epoch 1/3
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 71s 4ms/step - accuracy: 0.2034 - loss: 2.9748
Epoch 2/3
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 67s 4ms/step - accuracy: 0.2058 - loss: 2.9600
Epoch 3/3
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 67s 4ms/step - accuracy: 0.2095 - loss: 2.9474


2026/03/28 11:22:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_4_2013-01


In [6]:
from pathlib import Path

if mlflow_run_id is None:
    raise RuntimeError("MLflow run was not started. Please re-run the training cell.")

# Get experiment ID from the active run
active_run = mlflow.get_run(mlflow_run_id)
experiment_id = active_run.info.experiment_id

# Save both to file
output_path = Path('/kaggle/working/mlflow_ids.txt')
output_path.write_text(f"run_id={mlflow_run_id}\nexperiment_id={experiment_id}")

print(f"Saved run_id and experiment_id to {output_path}")
print("run_id:", mlflow_run_id)
print("experiment_id:", experiment_id)

Saved run_id and experiment_id to /kaggle/working/mlflow_ids.txt
run_id: d18e31a59e9041bd9a07774afda78fe0
experiment_id: 106643367198950269
